# Bath_dMFA_2026 Software 3

## Stock-flow-service nexus for vehicles - Stock-driven model

The passenger vehicle fleet of Europe evolves to meet future needs, replace old vehicles, and gradually transform to a low-carbon system. 

**This calculation exercise covers a future scenario for passenger-km by passenger vehicles, which is converted to a demand for an in-use stock (vehicle fleet size), which, by using a stock-driven model, is used to determine future new registration to maintain, expand, and transform the vehicle fleet.**

The data workbook *Bath_dMFA_2026_Software_Data.xlsx* contains a number of parameters for the calculation.

The result workbook *Bath_dMFA_2026_Software_2_Results.xlsx* contains the age-cohort structure of the existing fleet, which was derived during an earlier exercise and which is used here as an input.

The core of the vehicle fleet model is a description of the in-use stock of passenger vehicles (modelled as service providers, resource repository, consumption couplers, and dynamics determiners). Here, the inflow and the outflow of vehicles are determined from the historic fleet and the scenario time series of the stock (derived from given service level) via a stock-driven model and the lifetime of passenger vehicles in the use phase. Values are given at a per capita level and scaled up with the population. 

**Using the data provided in the file and generated from the previous exercise, calculate and interpret the following indicators/quantities:**

### Task 1: Load and inspect the required parameters: population, service, occupancy rate, and vehicle kilometrage! Import them as numpy arrays into the workspace. What is the shape of the four parameters and what is their dimension? Calculate the total stock needed in the future from the per capita service, population, and intensity of use parameters!

In [ ]:
# First, import required libraries:
import pandas as pd   # dataframe library, used for fast import/export from/to excel and for organizing the data.
import numpy as np    # math library
import matplotlib.pyplot as plt  # plotting library
import scipy # library with statistical functions
import pickle

In [ ]:
# Future population:
P_t = pd.read_excel('Bath_dMFA_2026_Software_Data.xlsx', sheet_name='Population', index_col=0).values[:,0]
P_t = 1e6 * P_t # convert unit from million to single people
print(P_t.shape) # rows: 36 years 2025-2060

In [ ]:
# Future annual passenger kilometers (mode: passenger vehicles)
# Tbd, add code here!

In [ ]:
# Future occupancy rate:
# Tbd, add code here!

In [ ]:
# Future annual vehicle kilometrage:
# Tbd, add code here!

With these parameters, we can calculate the required stock size, as derived in the lecture slides.

In [ ]:
S_future = np.einsum(# Tbd, add code here!

print(S_future.shape)
plt.plot(S_future)

In this scenario, the fleet size will slightly decline over time, as the decline in population outweights the increase in passenger-km/yr. 

This calculation concludes the first part of the stock-driven model, the calculation of the main driving variable (service to stock).

### Task 2: From the fleet size (stock) given for each future year, calculate inflow and outflow with a stock-driven model! Take into account the already existing stock and split the inflow into the different drive technologies!

Here, we first read the existing stock from the last workbook, the lifetime, and the future drive technology split. Then, we generate the pdf function from the lifetime, assuming a normally distributed lifetime with a standard deviation of 30% of the mean, a common lifetime model:

In [ ]:
# Historic stock (2024), by age-cohort, gasoline vehicles
S_tc_gasoline = pd.read_excel('Bath_dMFA_2026_Software_2_Results.xlsx', sheet_name='Stock_tc_gasoline', index_col=0).values
print(S_tc_gasoline.shape) # shape 35 years (rows) and age-cohorts (columns) 1990-2024

In [ ]:
# Historic stock (2024), by age-cohort, BEV
# Tbd, add code here!

In [ ]:
# Historic outflow (2024), by age-cohort, gasoline vehicles
# Tbd, add code here!

In [ ]:
# Historic outflow (2024), by age-cohort, BEVs
# Tbd, add code here!

In [ ]:
Lifetime_T = pd.read_excel('Bath_dMFA_2026_Software_Data.xlsx', sheet_name='Lifetime', index_col=0).values.transpose()
# For the subsequent calculation of the probability density function, the lifetime list must be a row vector, that's why the .transpose() command.
print(Lifetime_T.shape) # rows: aggregate for all age-cohorts, columns: 2 drive technologies

In [ ]:
I_c = pd.read_excel('Bath_dMFA_2026_Software_Data.xlsx', sheet_name='New registration', index_col=0).values[:,0]
print(I_c.shape) # rows: 35 years 1990-2024

In [ ]:
BEVshare_cT = pd.read_excel('Bath_dMFA_2026_Software_Data.xlsx', sheet_name='BEV_share', index_col=0).values
print(BEVshare_cT.shape) # selected rows: 71 years 1990-2060, columns: 2 drive technologies

In [ ]:
print(Lifetime_T)

pdf = scipy.stats.norm.pdf(np.arange(0,71,1),Lifetime_T,0.3*Lifetime_T) # pdf of the normal distribution with mean lifetime and standard deviation 0.3*lifetime

print(pdf.shape)

plt.plot(pdf.transpose()) # transpose to have time dimension on x axis, no fan.cy labels and legend here, just a quick look

We now program the stock-driven model.


In [ ]:
# The pdf indicates the probability of outflow of a given age-cohort.
# We first define the data arrays for inflow, stock by age-cohort, and outflow for the 71 years from 1990 to 2060
inflow_c    = np.zeros((71))
inflow_cT   = np.zeros((71,2))
stock_tcT   = np.zeros((71,71,2)) # stock table year by age-cohort and technology
outflow_tcT = np.zeros((71,71,2))

# Then, we put the historic values from the previous workbook into place:
inflow_c[0:35]    = I_c
inflow_cT[0:35,:] = np.einsum('c,cT->cT',I_c,BEVshare_cT[0:35,:]) / 100

stock_tcT[0:35,0:35,0] = S_tc_gasoline
stock_tcT[0:35,0:35,1] = S_tc_BEVs

outflow_tcT[0:35,0:35,0] = O_tc_gasoline
outflow_tcT[0:35,0:35,1] = O_tc_BEVs

# We now loop over all future years and both technologies:
for year in range(35,71):  # for all future years, starting in 2025
    # Preparation: Forward last year's stock to this year:
    stock_tcT[year,:,:]                = stock_tcT[year-1,:,:]
    for drivetech in range(0,2):       # for all drive technologies
        # First, the vehicle ageing:     calculate the outflow of the existing stock according to the lifetime model:
        veleave_t_c = np.zeros((71))   # empty cohort-specific outflow container for this year
        for age_cohort in range(0,year +1): # for all inflow years till year (need to add 1 since Python stops counting 1 place before the right boundary)
            # Calculate leaving vehicles in given year, for the different age-cohorts:
            veleave_t_c[age_cohort]    = inflow_cT[age_cohort,drivetech] * pdf[drivetech,year-age_cohort] 
        # Second: Add the leaving vehicles to outflow matrix and remove them from stock
        stock_tcT[year,:,drivetech]   -= veleave_t_c # remove leaving vehicles in place
        outflow_tcT[year,:,drivetech] += veleave_t_c # add leaving vehicles in place  
    # Third: Determine stock gap from balance between stock target and actual stock:
    Stock_rem                          = stock_tcT[year,:,:].sum() # REMaining stock in given year, both drive technologies combined
    Stock_gap                          = S_future[year-35] - Stock_rem # gap between stock target and actual stock, note the offset of 35 years, since the future stock starts in 2025.
    # Fourth: Calculate new inflow, split into drive technologies, and assign to stock
    inflow_c[year]                     = # Tbd, add code here!      # inflow is set to fill the gap
    inflow_cT[year,:]                  = # Tbd, add code here!
    # Fifth: Add new inflow to stock:
    stock_tcT[year,year,:]             = # Tbd, add code here! # diagonal of the stock (latest age-cohort) is the current inflow     
      
        

In [ ]:
# Check stock table of gasoline vehicles:
plt.imshow(stock_tcT[:,:,0],interpolation='nearest') # create a simple heatmap: larger value --> brighter color.  

In [ ]:
# Check stock table of BEVs:
# Tbd, add code here!

### Task 3: Show that inflow, stock, and outflow are balanced for vehicles in each year!


In [ ]:
# First, sum up over all age-cohorts, using the einsum function:
# Calculate aggreate system variables: stock_tT and outflow_tT:
stock_tT   = np.einsum('tcT->tT',stock_tcT)
outflow_tT = np.einsum('tcT->tT',outflow_tcT)

DStock_tT  = np.diff(stock_tT,axis = 0, prepend=0) # Calculate the stock change from the stock, starting with 0 in the first year (prepdent = 0)

Balance    = # Tbd, add code here!

In [ ]:
np.abs(Balance).sum() # add up all absolute values in this balance, should be 0 or at least close to it.

The overall added imbalance is about a millionth of a single car, from a practical perspective, the dynamic stock model is hence balanced.

In [ ]:
plt.plot(np.arange(1990,2061,1), inflow_cT) # create a simple line plot with the time series
plt.ylim(0, 1.1*np.max(inflow_cT))
plt.xlabel('age-cohort') # add x axis label 
plt.ylabel('new registration (inflow), cars') # add y axis label
plt.legend(['gasoline vehicles','battery electric vehicles']) # add a legend
plt.title('New registraction of passenger cars by age-cohort, EUR 20 countries') # add title
plt.show(); # show plot with all features from above

In [ ]:
plt.plot(# Tbd, add code here!

In this base scenario, the inflow or gasoline vehicles continues to decline and is replaced by BEVs, whose inflow rises to similar levels as we saw recently. Though the fleet size will be smaller in the future in this scenario, the shorter lifetime of BEVS (in this scenario) compensates for that, leading to new registration of similar size.

The outflow (scrapping) of end-of-life cars follows the inflow with a delay of about the lifetime of the cars, which means that the peak scrapping of gasoline vehicles has already passed and will decline to very small quantities in 2060. Scrap flows of BEVs will start rising from about 2030 onwards, surpassing 10 million units in about 2055.

### Task 4: Analyse the results of the stock-driven model:
**Task 4.1:** What is the ratio of BEVs in the new registration in 2020, 2030, 2040, 2050, and 2060, and how does it compare to the share of BEVs in the fleet in the same years?

To answer this question, we first slice the corresponding values from the respective system variable, by slicing it with a list of values: [30,40,50,60,70] corresponding to the years, 2020, 2030, 2040, 2050, and 2060.

In [ ]:
# For the share of BEVs in the inflow, the split is already given by the parameter BEVshare_cT:
BEVshare_cT[[30,40,50,60,70],1]

In [ ]:
# For the share in the stock, we slice and sum up the stock accordingly:
BEV_Stock_slice = stock_tT[[30,40,50,60,70],1]
Gas_Stock_slice = stock_tT[[30,40,50,60,70],0]
# print(BEV_Stock_slice)
# print(Gas_Stock_slice)
BEV_Stock_Share = # Tbd, add code here!
print (BEV_Stock_Share)

Due to the inertia of the fleet (in-use stocks as 'dynamics determiners'), the composition of the fleet slowly adapts to the composition of the new registration, but with a delay of 10-15 years, which corresponds to the average lifetime of the individual vehicle in the fleet.

Think of 2-3 other policy and decision-relevant question related to the composition of inflow, stock, and outflow, and calculate the corresponding indicators by slicing the variables accordingly!

### Task 5: export the core variables of the stock-driven model for further use!
We'll multiply age-cohort specific energy consumption and material composition to the system variables, so we'll export the stock and outflow tables by age-cohort.

In [ ]:
pd_xlsx_writer = pd.ExcelWriter('Bath_dMFA_2026_Software_3_Results.xlsx')

In [ ]:
export_df0= pd.DataFrame(data=inflow_cT,      # entire c x T inflow table for both drive technologies
            index  =np.arange(1990,2061,1),   # 1st column as index
            columns=['gasoline vehicles','battery electric vehicles'])   # 1st row as the column names

export_df0.to_excel(pd_xlsx_writer, sheet_name="Inflow_cT", merge_cells=False)  

export_df1= pd.DataFrame(data=stock_tcT[:,:,0],   # entire t x c stock table for gasoline cars
            index  =np.arange(1990,2061,1),   # 1st column as index
            columns=np.arange(1990,2061,1))   # 1st row as the column names

export_df1.to_excel(pd_xlsx_writer, sheet_name="Stock_tc_gasoline", merge_cells=False)   

export_df2= pd.DataFrame(data=stock_tcT[:,:,1],   # entire t x c stock table for BEVs
            index  =np.arange(1990,2061,1),   # 1st column as index
            columns=np.arange(1990,2061,1))   # 1st row as the column names

export_df2.to_excel(pd_xlsx_writer, sheet_name="Stock_tc_BEVs", merge_cells=False) 

export_df3= pd.DataFrame(data=outflow_tcT[:,:,0],   # entire t x c outflow table for gasoline cars
            index  =np.arange(1990,2061,1),   # 1st column as index
            columns=np.arange(1990,2061,1))   # 1st row as the column names

export_df3.to_excel(pd_xlsx_writer, sheet_name="Outflow_tc_gasoline", merge_cells=False)   

export_df4= pd.DataFrame(data=outflow_tcT[:,:,1],   # entire t x c outflow table for BEVs
            index  =np.arange(1990,2061,1),   # 1st column as index
            columns=np.arange(1990,2061,1))   # 1st row as the column names

export_df4.to_excel(pd_xlsx_writer, sheet_name="Outflow_tc_BEVs", merge_cells=False) 

In [ ]:
pd_xlsx_writer.close()

Finally, we'll use another export format, with a Python-internal formatting, to directly make the data available for the next workbook.

First, we use the Python dictionary data container to compile the different arrays into a single structure, and then, we 'pickle' it into a storage file for further use and easy loading.

In [ ]:
DynFleetResults = {'I_Veh_cT':inflow_cT, 'S_Veh_tcT':stock_tcT, 'O_Veh_tcT': outflow_tcT}

with open('Bath_dMFA_2026_Software_3_Results.pkl', 'wb') as f:  # open a text file
    pickle.dump(DynFleetResults, f) # serialize the list
f.close() 

This export concludes this exercise.